# MyDigitalTwin — Analyse Fréquentielle des Centres d'Intérêt

**Objectif** : Identifier les concepts les plus fréquents dans mes données réelles (Spotify, YouTube, Google, Netflix, Chrome) pour calibrer les macro-catégories de la home page.

**Approche** : Analyse de fréquence de mots par source → identification des gaps dans `CATEGORY_KEYWORDS` → mise à jour du dictionnaire.

> **Pourquoi pas K-Means ?**  
> Une première tentative de clustering TF-IDF + K-Means a produit un cluster *catch-all* dominant (~73% des données, Silhouette ≈ 0.19). Le problème est structurel : les textes courts multi-sources (artistes Spotify, titres YouTube, requêtes Google) ont des espaces sémantiques trop hétérogènes pour être clusterisés conjointement.  
> L'analyse fréquentielle est plus directe et interprétable pour des catégories prédéfinies.

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import sys, os

_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)

from pyspark.sql import functions as F
from config import build_spark_session, WAREHOUSE

spark = build_spark_session("MyDigitalTwin-FrequencyAnalysis", delta=True)
spark.sparkContext.setLogLevel("WARN")

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(table_name):
    return spark.read.format("delta").load(os.path.join(WAREHOUSE, table_name))

Warehouse: /opt/spark/data/warehouse


26/05/12 17:20:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## Étape 1 — Chargement des sources texte

On extrait la colonne texte pertinente de chaque source Delta, en gardant la provenance (`source`) pour analyser chaque canal séparément.

In [2]:
# ── 1. CHARGEMENT ─────────────────────────────────────────────────────────────
sources = {
    "Google Searches": read_table("google_searches").select(F.col("query").alias("text")),
    "YouTube":         read_table("youtube_watch").select(F.col("title").alias("text")),
    "Chrome":          read_table("google_chrome").select(F.col("title").alias("text")),
    "Spotify":         read_table("spotify_streams").select(F.col("artistName").alias("text")).dropDuplicates(["text"]),
    "Netflix":         read_table("netflix_views").select(F.col("show_title").alias("text")),
}

for name, df in sources.items():
    print(f"{name:20s}: {df.count():>6,} lignes")

26/05/12 17:20:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Google Searches     : 55,827 lignes
YouTube             : 14,076 lignes
Chrome              :    338 lignes


Spotify             :  5,926 lignes
Netflix             :  4,288 lignes


In [3]:
# ── 2. ANALYSE FRÉQUENTIELLE PAR SOURCE ───────────────────────────────────────
from pyspark.ml.feature import Tokenizer, StopWordsRemover

# Stopwords : anglais + français + bruit technique
STOPWORDS_EXTRA = [
    # Français
    "les", "des", "une", "sur", "avec", "dans", "qui", "que", "par", "plus",
    "tout", "bien", "comme", "mais", "mon", "ton", "son", "nos", "mes",
    "faire", "comment", "plus", "aussi", "encore", "très",
    # Anglais générique
    "the", "and", "for", "with", "you", "your", "this", "that", "from",
    "are", "was", "not", "its", "but", "all", "new", "best", "how",
    # Bruit technique (URLs, tracking, ads)
    "https", "http", "www", "com", "org", "net", "html", "php", "utm",
    "amp", "utm_source", "befr", "dgoogle", "watch", "video", "clip",
    "official", "officiel", "youtube", "shorts",
]

STOP_ALL = StopWordsRemover.loadDefaultStopWords("english") \
         + StopWordsRemover.loadDefaultStopWords("french") \
         + STOPWORDS_EXTRA

results = {}

for source_name, df in sources.items():
    clean = df.filter(
        F.col("text").isNotNull() &
        (F.length(F.col("text")) > 2) &
        (~F.col("text").rlike(r'^https?://'))   # exclure les URLs brutes
    )

    tokenized = Tokenizer(inputCol="text", outputCol="words").transform(clean)
    filtered  = StopWordsRemover(
        inputCol="words", outputCol="tokens", stopWords=STOP_ALL
    ).transform(tokenized)

    freq = (
        filtered
        .select(F.explode("tokens").alias("word"))
        .filter(F.length("word") > 2)
        .groupBy("word")
        .count()
        .orderBy(F.desc("count"))
    )

    results[source_name] = freq
    print(f"✓ {source_name}")

print("\nAnalyse terminée.")

✓ Google Searches
✓ YouTube
✓ Chrome
✓ Spotify
✓ Netflix

Analyse terminée.


In [4]:
# ── 3. TOP 25 MOTS PAR SOURCE ─────────────────────────────────────────────────
TOP_N = 25

for source_name, freq_df in results.items():
    print(f"\n{'='*50}")
    print(f"  {source_name}")
    print(f"{'='*50}")
    freq_df.show(TOP_N, truncate=False)


  Google Searches


+----------+-----+
|word      |count|
+----------+-----+
|belgique  |309  |
|one       |268  |
|fifa      |261  |
|google    |240  |
|streaming |229  |
|piece     |208  |
|minecraft |204  |
|%c3%a0    |189  |
|prix      |178  |
|hannut    |159  |
|fortnite  |159  |
|synonyme  |154  |
|musique   |150  |
|mac       |144  |
|discord   |144  |
|pdf       |139  |
|film      |137  |
|carte     |137  |
|download  |137  |
|traduction|136  |
|pro       |135  |
|mp3       |132  |
|font      |131  |
|france    |129  |
|club      |126  |
+----------+-----+
only showing top 25 rows


  YouTube
+---------+-----+
|word     |count|
+---------+-----+
|16x9     |615  |
|inh      |523  |
|(clip    |284  |
|officiel)|284  |
|mix      |263  |
|live     |218  |
|(official|215  |
|vid      |213  |
|video)   |206  |
|1920x1080|192  |
|15s      |191  |
|music    |188  |
|house    |178  |
|2024     |177  |
|2025     |175  |
|(ft      |165  |
|ft.      |164  |
|squeezie |159  |
|one      |142  |
|#shorts  |130  

In [5]:
# ── 4. BIGRAMMES — Termes composés importants ─────────────────────────────────
# Les bigrammes capturent des concepts que les mots seuls manquent :
# "travis scott", "league of legends", "formula 1", "deep learning", etc.
from pyspark.ml.feature import NGram

bigram_results = {}

for source_name, df in sources.items():
    clean = df.filter(
        F.col("text").isNotNull() &
        (F.length(F.col("text")) > 2) &
        (~F.col("text").rlike(r'^https?://'))
    )

    tokenized = Tokenizer(inputCol="text", outputCol="words").transform(clean)
    filtered  = StopWordsRemover(
        inputCol="words", outputCol="tokens", stopWords=STOP_ALL
    ).transform(tokenized)

    bigrams = NGram(n=2, inputCol="tokens", outputCol="ngrams").transform(filtered)

    freq = (
        bigrams
        .select(F.explode("ngrams").alias("bigram"))
        .filter(F.length("bigram") > 5)
        .groupBy("bigram")
        .count()
        .orderBy(F.desc("count"))
    )

    bigram_results[source_name] = freq

print("Top bigrammes par source :\n")
for source_name, freq_df in bigram_results.items():
    print(f"── {source_name}")
    freq_df.show(15, truncate=False)
    print()

Top bigrammes par source :

── Google Searches
+---------------+-----+
|bigram         |count|
+---------------+-----+
|one piece      |194  |
|fifa 21        |148  |
|fifa 23        |59   |
|ralph lauren   |46   |
|streaming vf   |42   |
|travis scott   |42   |
|airpods pro    |39   |
|epic games     |39   |
|google flight  |37   |
|freeze corleone|37   |
|virtual dj     |36   |
|rocket league  |35   |
|stg gege       |33   |
|polo ralph     |33   |
|star wars      |32   |
+---------------+-----+
only showing top 15 rows


── YouTube
+--------------------+-----+
|bigram              |count|
+--------------------+-----+
|(clip officiel)     |280  |
|16x9 6s             |264  |
|vid 16x9            |195  |
|inh inh             |195  |
|choisissez chrome   |110  |
|- rediffusion       |110  |
|rediffusion squeezie|110  |
|inh cards           |106  |
|travis scott        |103  |
|inazuma eleven      |97   |
|music video)        |90   |
|(official video)    |86   |
|(official music     |85

In [6]:
# ── 5. GAP ANALYSIS — Termes fréquents non couverts par CATEGORY_KEYWORDS ─────
# CATEGORY_KEYWORDS est défini dans config.yaml — modifier là-bas pour personnaliser.
from config import CATEGORY_KEYWORDS

all_kw = {kw for kws in CATEGORY_KEYWORDS.values() for kw in kws}

# Union de toutes les sources pour la vue globale
from functools import reduce
all_sources = reduce(lambda a, b: a.union(b), sources.values())
clean_all = all_sources.filter(
    F.col("text").isNotNull() &
    (F.length(F.col("text")) > 2) &
    (~F.col("text").rlike(r'^https?://'))
)
tokenized_all = Tokenizer(inputCol="text", outputCol="words").transform(clean_all)
filtered_all  = StopWordsRemover(inputCol="words", outputCol="tokens", stopWords=STOP_ALL).transform(tokenized_all)

global_freq = (
    filtered_all
    .select(F.explode("tokens").alias("word"))
    .filter(F.length("word") > 2)
    .groupBy("word").count()
    .orderBy(F.desc("count"))
)

top_words = [row["word"] for row in global_freq.limit(200).collect()]
uncovered = [w for w in top_words if w not in all_kw]

print("Top 40 mots fréquents NON couverts par CATEGORY_KEYWORDS :")
print("(candidats à ajouter dans une catégorie)")
for w in uncovered[:40]:
    print(f"  {w}")

Top 40 mots fréquents NON couverts par CATEGORY_KEYWORDS :
(candidats à ajouter dans une catégorie)
  16x9
  inh
  one
  piece
  google
  (clip
  officiel)
  mix
  pro
  live
  2024
  (official
  vid
  club
  prix
  squeezie
  video)
  2025
  1920x1080
  15s
  %c3%a0
  mac
  saison
  musique
  black
  download
  ft.
  (ft
  hannut
  jbl
  show
  travis
  synonyme
  jeu
  scott
  fairy
  tail
  (2011)
  carte
  chrome


In [9]:
spark.stop()
print("Spark session fermée.")

Spark session fermée.
